# Parakeet Fine-Tuning — NeMo Environment Setup (Colab)

Run top-to-bottom on a Colab **GPU** runtime (Runtime > Change runtime type > GPU).

## 1. Install system & Python dependencies

In [ ]:
!apt-get update -qq && apt-get install -y -qq sox libsndfile1 ffmpeg libsox-fmt-mp3
!pip install -q text-unidecode Cython "matplotlib>=3.3.2"

## 2. Install NeMo toolkit (ASR)

Colab may prompt you to **restart the runtime** after this finishes — do that, then continue running from the next cell (no need to rerun the installs).

In [ ]:
!pip install -q "nemo_toolkit[asr]" lightning hydra-core omegaconf soundfile

## 3. Verify environment (GPU, NeMo, PyTorch)

In [ ]:
import torch
import nemo
import lightning.pytorch as pl

print("torch     :", torch.__version__)
print("CUDA      :", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU       :", torch.cuda.get_device_name(0))
print("nemo      :", nemo.__version__)
print("lightning :", pl.__version__)

## 4. Download & prepare tutorial data (AN4)

This is NVIDIA's standard AN4 smoke-test dataset — it confirms the download/convert/manifest pipeline works end-to-end on this runtime before pointing it at the real dataset. For the actual project data, use `prepare_manifest.py --dataset-json dataset.json --val-split ...` instead of this section.

In [ ]:
import urllib.request

url = "https://dldata-public.s3.us-east-2.amazonaws.com/an4_sphere.tar.gz"
urllib.request.urlretrieve(url, "an4_sphere.tar.gz")

print("Download complete. Extracting files...")

In [ ]:
import os
import tarfile

# Current working directory
DATA_DIR = os.getcwd()
os.environ["DATA_DIR"] = DATA_DIR

# Extract the tar.gz file
with tarfile.open("an4_sphere.tar.gz", "r:gz") as tar:
    tar.extractall(path=DATA_DIR)

print("Extraction complete!")

# Verify the an4 folder exists
an4_path = os.path.join(DATA_DIR, "an4")
print("AN4 path:", an4_path)

In [ ]:
import json
import librosa
import os
import glob
import subprocess

source_data_dir = f"{DATA_DIR}/an4"
target_data_dir = f"{DATA_DIR}/an4_converted"


def an4_build_manifest(transcripts_path, manifest_path, target_wavs_dir):
    """Build AN4 manifest"""

    with open(transcripts_path, "r") as fin:
        with open(manifest_path, "w") as fout:

            for line in fin:
                transcript = line[: line.find("(") - 1].lower()
                transcript = transcript.replace("<s>", "")
                transcript = transcript.replace("</s>", "")
                transcript = transcript.strip()

                file_id = line[line.find("(") + 1 : -2]

                audio_path = os.path.join(
                    target_wavs_dir,
                    file_id + ".wav"
                )

                duration = librosa.get_duration(path=audio_path)

                metadata = {
                    "audio_filepath": audio_path,
                    "duration": duration,
                    "text": transcript,
                }

                json.dump(metadata, fout)
                fout.write("\n")


# ------------------------------------------------------------------
# Check dataset
# ------------------------------------------------------------------
if not os.path.exists(source_data_dir):
    raise ValueError(f"Dataset not found: {source_data_dir}")

# ------------------------------------------------------------------
# Find SPH files
# ------------------------------------------------------------------
sph_list = glob.glob(
    os.path.join(source_data_dir, "**", "*.sph"),
    recursive=True
)

print(f"Found {len(sph_list)} SPH files")

# ------------------------------------------------------------------
# WAV output directory
# ------------------------------------------------------------------
target_wavs_dir = os.path.join(target_data_dir, "wavs")
os.makedirs(target_wavs_dir, exist_ok=True)

# ------------------------------------------------------------------
# Skip conversion if WAV files already exist
# ------------------------------------------------------------------
existing_wavs = glob.glob(
    os.path.join(target_wavs_dir, "*.wav")
)

if len(existing_wavs) > 0:
    print(f"✅ Found {len(existing_wavs)} existing WAV files.")
    print("✅ Skipping audio conversion.")
else:
    print("Converting SPH files to WAV...")

    for sph_path in sph_list:

        wav_path = os.path.join(
            target_wavs_dir,
            os.path.splitext(
                os.path.basename(sph_path)
            )[0] + ".wav"
        )

        subprocess.run(
            [
                "ffmpeg",
                "-y",
                "-i",
                sph_path,
                wav_path,
            ],
            stdout=subprocess.DEVNULL,
            stderr=subprocess.DEVNULL,
            check=True,
        )

    print("✅ Audio conversion completed.")

# ------------------------------------------------------------------
# Build training manifest
# ------------------------------------------------------------------
train_transcripts = os.path.join(
    source_data_dir,
    "etc",
    "an4_train.transcription"
)

train_manifest = os.path.join(
    target_data_dir,
    "train_manifest.json"
)

an4_build_manifest(
    train_transcripts,
    train_manifest,
    target_wavs_dir
)

# ------------------------------------------------------------------
# Build test manifest
# ------------------------------------------------------------------
test_transcripts = os.path.join(
    source_data_dir,
    "etc",
    "an4_test.transcription"
)

test_manifest = os.path.join(
    target_data_dir,
    "test_manifest.json"
)

an4_build_manifest(
    test_transcripts,
    test_manifest,
    target_wavs_dir
)

print("\n✅ Done!")
print("Train manifest:", train_manifest)
print("Test manifest :", test_manifest)

## 5. Sanity-check a converted sample

In [ ]:
# change path of the file here
import os
import IPython.display as ipd
path = os.environ["DATA_DIR"] + '/an4_converted/wavs/an268-mbmg-b.wav'
ipd.Audio(path)

## 6. Fine-tune parakeet-tdt-0.6b-v3 (smoke test on AN4)

Uses `finetune_parakeet.py` from this repo instead of cloning the full NVIDIA/NeMo source tree — it's model-agnostic (handles parakeet's TDT joint correctly, unlike the stock hybrid-CTC tutorial config) and already has fine-tuning-appropriate defaults (`lr=1e-5`, not the `0.1` used for near-scratch training).

This is a **pipeline smoke test on the toy AN4 data** — `max_epochs=1` and small batch size are just to prove everything runs end-to-end on this GPU without OOMing. It will not produce a good model. Swap in the real dataset once this passes clean (see Next steps).

In [ ]:
!git clone -q https://github.com/XXoussam/Fine-tune-parakeet.git "$DATA_DIR/Fine-tune-parakeet"
%cd "$DATA_DIR/Fine-tune-parakeet"

!python finetune_parakeet.py \
    model.train_ds.manifest_filepath="$DATA_DIR/an4_converted/train_manifest.json" \
    model.validation_ds.manifest_filepath="$DATA_DIR/an4_converted/test_manifest.json" \
    model.train_ds.batch_size=4 \
    model.validation_ds.batch_size=4 \
    trainer.max_epochs=1 \
    exp_manager.exp_dir="$DATA_DIR/checkpoints"

## 7. Full fine-tune (still on AN4 data)

Same data and the same `finetune_parakeet.py`, just the full training schedule instead of the 1-epoch smoke test above: `batch_size` and `max_epochs` overrides are dropped so it falls back to the `conf/parakeet_finetune.yaml` defaults (`batch_size=16`, `max_epochs=50`) — raise `batch_size` further if the GPU has VRAM to spare.

Writes to a separate `exp_dir` (`checkpoints_full`, not `checkpoints`) so it starts fresh from the pretrained checkpoint instead of resuming the smoke test's 1-epoch run.

AN4 is small and narrow-vocabulary, so a great WER here isn't proof the model generalizes — this run validates that a full-length schedule completes cleanly (checkpointing, no OOM, no mid-schedule crash). The real signal comes once you swap in the real dataset (see Next steps).

In [ ]:
!python finetune_parakeet.py \
    model.train_ds.manifest_filepath="$DATA_DIR/an4_converted/train_manifest.json" \
    model.validation_ds.manifest_filepath="$DATA_DIR/an4_converted/test_manifest.json" \
    exp_manager.exp_dir="$DATA_DIR/checkpoints_full"

## Next steps

- Once the full AN4 run above completes cleanly, swap the manifests for the real dataset: `python prepare_manifest.py --dataset-json dataset.json --val-split 0.1 --train-out train_manifest.json --val-out val_manifest.json`, then re-run `finetune_parakeet.py` with `model.train_ds.manifest_filepath=...` pointed at it.
- Its NeMo API calls haven't yet been verified against nemo 3.0.0 specifically — the smoke test and full run above are that verification.